INSTALL AND IMPORT

In [2]:
!pip install boto3 -q
import boto3
from google.colab import userdata

SET UP S3 CLIENT

In [4]:
AWS_ACCESS_KEY_ID = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
AWS_REGION = 'ap-south-1'

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

Verifying Connection

In [9]:
BUCKET_NAME = 'akshita-flight-delay-2026'

response = s3.list_objects_v2(Bucket=BUCKET_NAME)
for obj in response.get('Contents', []):
    print(obj['Key'], '-', obj['Size'], 'bytes')

flights_sample_100k.csv - 20477042 bytes


CSV from S3 to pandas

In [11]:
import pandas as pd
from io import BytesIO

FILE_KEY = 'flights_sample_100k.csv'

obj = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_KEY)
df = pd.read_csv(BytesIO(obj['Body'].read()))

print(df.shape)
df.head()

(100000, 32)


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-03-01,Allegiant Air,Allegiant Air: G4,G4,20368,1668,PGD,"Punta Gorda, FL",SPI,"Springfield, IL",...,0.0,160.0,138.0,122.0,994.0,NaN,NaN,NaN,NaN,NaN
1,2021-02-16,American Airlines Inc.,American Airlines Inc.: AA,AA,19805,2437,DFW,"Dallas/Fort Worth, TX",LAX,"Los Angeles, CA",...,0.0,211.0,NaN,NaN,1235.0,NaN,NaN,NaN,NaN,NaN
2,2022-04-12,PSA Airlines Inc.,PSA Airlines Inc.: OH,OH,20397,5560,EWN,"New Bern/Morehead/Beaufort, NC",CLT,"Charlotte, NC",...,0.0,79.0,78.0,51.0,221.0,NaN,NaN,NaN,NaN,NaN
3,2021-10-13,Southwest Airlines Co.,Southwest Airlines Co.: WN,WN,19393,1944,ABQ,"Albuquerque, NM",DEN,"Denver, CO",...,0.0,80.0,71.0,49.0,349.0,10.0,0.0,0.0,0.0,6.0
4,2022-06-05,Southwest Airlines Co.,Southwest Airlines Co.: WN,WN,19393,3081,PIT,"Pittsburgh, PA",STL,"St. Louis, MO",...,0.0,105.0,100.0,82.0,554.0,NaN,NaN,NaN,NaN,NaN


Install awswrangler

In [16]:
!pip install awswrangler -q
import awswrangler as wr

# Set up session with your existing credentials
import boto3
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.3/396.3 kB 7.6 MB/s eta 0:00:00


Pull data straight from Athena into pandas

In [17]:
query = """
SELECT *
FROM flight_delay_db.flights_100k
"""

df = wr.athena.read_sql_query(
    sql=query,
    database="flight_delay_db",
    boto3_session=session
)

print(df.shape)
df.head()

/usr/local/lib/python3.13/dist-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


(100000, 32)


,fl_date,airline,airline_dot,airline_code,dot_code,fl_number,origin,origin_city,dest,dest_city,...,diverted,crs_elapsed_time,elapsed_time,air_time,distance,delay_due_carrier,delay_due_weather,delay_due_nas,delay_due_security,delay_due_late_aircraft
0,2019-03-01,Allegiant Air,Allegiant Air: G4,G4,20368,1668,PGD,"Punta Gorda, FL",SPI,"Springfield, IL",...,0.0,160.0,138.0,122.0,994.0,NaN,NaN,NaN,NaN,NaN
1,2021-02-16,American Airlines Inc.,American Airlines Inc.: AA,AA,19805,2437,DFW,"Dallas/Fort Worth, TX",LAX,"Los Angeles, CA",...,0.0,211.0,NaN,NaN,1235.0,NaN,NaN,NaN,NaN,NaN
2,2022-04-12,PSA Airlines Inc.,PSA Airlines Inc.: OH,OH,20397,5560,EWN,"New Bern/Morehead/Beaufort, NC",CLT,"Charlotte, NC",...,0.0,79.0,78.0,51.0,221.0,NaN,NaN,NaN,NaN,NaN
3,2021-10-13,Southwest Airlines Co.,Southwest Airlines Co.: WN,WN,19393,1944,ABQ,"Albuquerque, NM",DEN,"Denver, CO",...,0.0,80.0,71.0,49.0,349.0,10.0,0.0,0.0,0.0,6.0
4,2022-06-05,Southwest Airlines Co.,Southwest Airlines Co.: WN,WN,19393,3081,PIT,"Pittsburgh, PA",STL,"St. Louis, MO",...,0.0,105.0,100.0,82.0,554.0,NaN,NaN,NaN,NaN,NaN


 Create binary target

In [19]:
print(df.columns.tolist())

['fl_date', 'airline', 'airline_dot', 'airline_code', 'dot_code', 'fl_number', 'origin', 'origin_city', 'dest', 'dest_city', 'crs_dep_time', 'dep_time', 'dep_delay', 'taxi_out', 'wheels_off', 'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay', 'cancelled', 'cancellation_code', 'diverted', 'crs_elapsed_time', 'elapsed_time', 'air_time', 'distance', 'delay_due_carrier', 'delay_due_weather', 'delay_due_nas', 'delay_due_security', 'delay_due_late_aircraft']


In [20]:
df.columns = df.columns.str.upper()  # convert back to match what we've been using

df['IS_DELAYED'] = (df['DEP_DELAY'] > 15).astype(int)

print(df['IS_DELAYED'].value_counts())
print(df['IS_DELAYED'].value_counts(normalize=True))

IS_DELAYED
0    82750
1    17250
Name: count, dtype: int64
IS_DELAYED
0    0.8275
1    0.1725
Name: proportion, dtype: float64


Feature engineering

In [21]:
import pandas as pd

# Extract date-based features
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])
df['MONTH'] = df['FL_DATE'].dt.month
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek  # 0=Monday, 6=Sunday

# Extract hour from scheduled departure time (CRS_DEP_TIME is like 1430 for 2:30 PM)
df['DEP_HOUR'] = (df['CRS_DEP_TIME'] // 100).astype(int)

# Select features for modeling — dropping columns that would leak the answer
# (e.g. ARR_DELAY, TAXI_OUT are only known AFTER departure — can't use them to predict delay)
features = ['AIRLINE', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'DISTANCE']
target = 'IS_DELAYED'

model_df = df[features + [target]].dropna()
print(model_df.shape)
model_df.head()

(100000, 8)


,AIRLINE,ORIGIN,DEST,MONTH,DAY_OF_WEEK,DEP_HOUR,DISTANCE,IS_DELAYED
0,Allegiant Air,PGD,SPI,3,4,6,994.0,0
1,American Airlines Inc.,DFW,LAX,2,1,13,1235.0,0
2,PSA Airlines Inc.,EWN,CLT,4,1,6,221.0,0
3,Southwest Airlines Co.,ABQ,DEN,10,2,17,349.0,1
4,Southwest Airlines Co.,PIT,STL,6,6,5,554.0,0


Encode categoricals and split data

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

model_df = model_df.copy()

# Label encode ORIGIN and DEST (too many unique values for one-hot encoding)
for col in ['AIRLINE', 'ORIGIN', 'DEST']:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col])

X = model_df.drop(columns=['IS_DELAYED'])
y = model_df['IS_DELAYED']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(80000, 7) (20000, 7)


Train XGBoost


In [23]:
!pip install xgboost -q
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# scale_pos_weight compensates for the class imbalance (roughly 82:17 ratio)
scale = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.88      0.67      0.76     16550
           1       0.26      0.57      0.36      3450

    accuracy                           0.65     20000
   macro avg       0.57      0.62      0.56     20000
weighted avg       0.77      0.65      0.69     20000

ROC-AUC: 0.6591663908227156


Add historical delay-rate features

In [24]:

model_df = model_df.copy()

# Average delay rate per airline
airline_delay_rate = model_df.groupby('AIRLINE')['IS_DELAYED'].mean()
model_df['AIRLINE_DELAY_RATE'] = model_df['AIRLINE'].map(airline_delay_rate)

# Average delay rate per origin airport
origin_delay_rate = model_df.groupby('ORIGIN')['IS_DELAYED'].mean()
model_df['ORIGIN_DELAY_RATE'] = model_df['ORIGIN'].map(origin_delay_rate)

# Average delay rate per hour of day (some hours are just more delay-prone — cascading delays later in the day)
hour_delay_rate = model_df.groupby('DEP_HOUR')['IS_DELAYED'].mean()
model_df['HOUR_DELAY_RATE'] = model_df['DEP_HOUR'].map(hour_delay_rate)

print(model_df.shape)
model_df.head()

(100000, 11)


,AIRLINE,ORIGIN,DEST,MONTH,DAY_OF_WEEK,DEP_HOUR,DISTANCE,IS_DELAYED,AIRLINE_DELAY_RATE,ORIGIN_DELAY_RATE,HOUR_DELAY_RATE
0,1,266,336,3,4,6,994.0,0,0.215146,0.176471,0.069267
1,2,97,197,2,1,13,1235.0,0,0.182150,0.200945,0.183876
2,12,119,76,4,1,6,221.0,0,0.157837,0.238095,0.069267
3,15,2,98,10,2,17,349.0,1,0.213786,0.146179,0.230781
4,15,275,341,6,6,5,554.0,0,0.213786,0.132686,0.070992


Re-split with new features


In [25]:
X = model_df.drop(columns=['IS_DELAYED'])
y = model_df['IS_DELAYED']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale, eval_metric='logloss', random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = xgb_model.predict(X_test)

print("=== XGBoost ===")
print(classification_report(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_proba))

# LightGBM
!pip install lightgbm -q
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale, random_state=42, verbose=-1
)
lgbm_model.fit(X_train, y_train)
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]
lgbm_pred = lgbm_model.predict(X_test)

print("\n=== LightGBM ===")
print(classification_report(y_test, lgbm_pred))
print("ROC-AUC:", roc_auc_score(y_test, lgbm_proba))

=== XGBoost ===
              precision    recall  f1-score   support

           0       0.88      0.67      0.76     16550
           1       0.26      0.57      0.36      3450

    accuracy                           0.65     20000
   macro avg       0.57      0.62      0.56     20000
weighted avg       0.77      0.65      0.69     20000

ROC-AUC: 0.6625758570865625

=== LightGBM ===
              precision    recall  f1-score   support

           0       0.88      0.65      0.75     16550
           1       0.26      0.59      0.36      3450

    accuracy                           0.64     20000
   macro avg       0.57      0.62      0.56     20000
weighted avg       0.78      0.64      0.68     20000

ROC-AUC: 0.6657163360917728
